# Task 5: Traffic Recommendation System

## 5.1 Objective

The objective of this task is to transform the traffic analysis and model outputs into a practical traffic recommendation system.

Because the dataset represents a single traffic corridor, the system focuses on recommending suitable travel times rather than selecting alternative routes.

The recommendation system will:

- Identify lower-traffic travel periods.
- Consider weekday and weekend travel patterns.
- Consider weather conditions where appropriate.
- Recommend suitable travel windows based on historical traffic patterns.
- Generate clear, plain-language recommendations for users.

The recommendations are intended to support travel planning by identifying periods when traffic volumes are typically lower.

In [2]:
# ============================================================
# 5.2 Data Loading and Preparation
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

In C:\Users\tanak\anaconda3\Lib\site-packages\matplotlib\mpl-data\stylelib\bmh.mplstyle: The text.hinting_factor rcParam was deprecated in Matplotlib 3.11 and will be removed in 3.13.
In C:\Users\tanak\anaconda3\Lib\site-packages\matplotlib\mpl-data\stylelib\classic.mplstyle: The text.hinting_factor rcParam was deprecated in Matplotlib 3.11 and will be removed in 3.13.
In C:\Users\tanak\anaconda3\Lib\site-packages\matplotlib\mpl-data\stylelib\_classic_test_patch.mplstyle: The text.kerning_factor rcParam was deprecated in Matplotlib 3.11 and will be removed in 3.13.


Libraries imported successfully.


In [4]:
# ============================================================
# 5.2 Load and Validate Engineered Traffic Data
# ============================================================

from pathlib import Path
import pandas as pd

# Reuse the engineered dataset produced in Part 2.
# This keeps Task 5 consistent with the earlier project pipeline.
DATA_PATH = Path("../part2_python/engineered_traffic_data.csv")

required_columns = {
    "date_time",
    "traffic_volume",
    "hour",
    "day_of_week",
    "is_weekend",
    "weather_main",
    "weather_description",
    "has_rain",
    "has_snow",
    "congestion_category"
}

try:
    df = pd.read_csv(
        DATA_PATH,
        parse_dates=["date_time"]
    )

    missing_columns = required_columns - set(df.columns)

    if missing_columns:
        raise ValueError(
            f"Missing required columns: {sorted(missing_columns)}"
        )

    print("Engineered traffic dataset loaded successfully.")
    print(f"Dataset shape: {df.shape}")
    print(f"Date range: {df['date_time'].min()} to {df['date_time'].max()}")
    print(f"Required Task 5 features available: {len(required_columns)}/{len(required_columns)}")

except FileNotFoundError:
    print(f"ERROR: Dataset not found at: {DATA_PATH}")
    print("Check the relative path to engineered_traffic_data.csv.")

except (ValueError, pd.errors.ParserError) as error:
    print(f"ERROR: {error}")

Engineered traffic dataset loaded successfully.
Dataset shape: (48187, 30)
Date range: 2012-10-02 09:00:00 to 2018-09-30 23:00:00
Required Task 5 features available: 10/10


## 5.3 Identify Lower-Traffic Travel Periods

The first stage of the recommendation system is to identify historical periods with relatively low traffic volume.

Because traffic behaviour may differ between working days and weekends, the data is separated into **Weekday** and **Weekend** observations using the engineered `is_weekend` feature created in Part 2.

For each day type, the system:

1. Groups historical observations by hour of day.
2. Calculates the mean traffic volume for each hour.
3. Records the number of observations supporting each estimate.
4. Ranks the hourly periods from lowest to highest average traffic volume.
5. Identifies the five historically lowest-traffic hours for weekdays and weekends.

These results provide an evidence-based starting point for the recommendation system. However, the lowest-volume hour is not automatically treated as the best recommendation because very early-morning periods may have low traffic but limited practical value for typical travellers. Later stages therefore combine historical traffic patterns with practical travel windows and weather conditions.

In [5]:
# ============================================================
# 5.3 Identify Lower-Traffic Travel Periods
# ============================================================

# Create a readable day-type label
df["day_type"] = np.where(
    df["is_weekend"] == 1,
    "Weekend",
    "Weekday"
)

# Calculate historical average traffic volume by day type and hour
hourly_traffic = (
    df.groupby(["day_type", "hour"], as_index=False)
      .agg(
          avg_traffic_volume=("traffic_volume", "mean"),
          observations=("traffic_volume", "size")
      )
)

# Rank hours from lowest to highest traffic within each day type
hourly_traffic["traffic_rank"] = (
    hourly_traffic
    .groupby("day_type")["avg_traffic_volume"]
    .rank(method="dense")
    .astype(int)
)

# Display the five lowest-traffic hours for each day type
lowest_traffic_periods = (
    hourly_traffic
    .sort_values(
        ["day_type", "avg_traffic_volume"]
    )
    .groupby("day_type")
    .head(5)
    .reset_index(drop=True)
)

print("Five lowest-traffic hours by day type:")
print()

for day_type in ["Weekday", "Weekend"]:
    print(f"{day_type}:")
    
    result = lowest_traffic_periods[
        lowest_traffic_periods["day_type"] == day_type
    ][
        ["hour", "avg_traffic_volume", "observations"]
    ].copy()

    result["avg_traffic_volume"] = (
        result["avg_traffic_volume"].round(1)
    )

    print(result.to_string(index=False))
    print()

Five lowest-traffic hours by day type:

Weekday:
 hour  avg_traffic_volume  observations
    2               302.0          1455
    3               362.3          1456
    1               396.9          1449
    0               651.7          1466
    4               832.7          1496

Weekend:
 hour  avg_traffic_volume  observations
    4               375.4           595
    3               393.6           569
    2               611.2           564
    5               639.2           607
    1               805.7           598



### Interpretation of Historical Low-Traffic Periods

The analysis shows a clear concentration of the lowest historical traffic volumes during overnight and early-morning hours.

For **weekdays**, the lowest average traffic volume occurs at **02:00**, with approximately **302 vehicles per hour**. The next-lowest periods are 03:00 and 01:00, followed by 00:00 and 04:00.

For **weekends**, the lowest average traffic volume occurs at **04:00**, with approximately **375 vehicles per hour**. Other low-volume periods are also concentrated between 01:00 and 05:00.

Each hourly estimate is supported by several hundred to more than one thousand historical observations, providing a substantial historical basis for the comparison.

However, simply recommending the hour with the minimum traffic volume would produce mostly overnight recommendations. Although these periods are objectively low-traffic, they may not be practical for typical daytime travellers.

Therefore, the recommendation system will distinguish between:

- **Historical minimum-traffic periods**, which describe the overall traffic pattern.
- **Practical travel windows**, which identify relatively low-traffic periods within more useful daytime and evening travel hours.

This distinction makes the recommendation system more actionable while ensuring that its recommendations remain grounded in observed traffic data.

## 5.4 Identify Practical Travel Windows

The historical analysis identified the lowest traffic volumes mainly during overnight and early-morning hours. Although these periods represent the absolute minimum traffic levels, they may not be practical recommendations for many travellers.

To make the recommendation system more useful, a **practical travel period from 06:00 to 22:00** is used for the initial analysis. This range covers typical daytime and evening travel while excluding the very low-volume overnight period identified previously.

Within this practical period, the system:

1. Separates weekday and weekend traffic patterns.
2. Calculates the historical mean traffic volume for each hour.
3. Compares each hourly average with the overall average for the same day type.
4. Calculates the percentage reduction in traffic volume.
5. Identifies the lowest-traffic practical travel periods.

The 06:00–22:00 range is used as an analytical default rather than a universal assumption. The final recommendation function will allow the user to specify a preferred travel window so that recommendations can be adapted to individual travel needs.

In [6]:
# ============================================================
# 5.4 Identify Practical Travel Windows
# ============================================================

PRACTICAL_START_HOUR = 6
PRACTICAL_END_HOUR = 22

# Keep only practical daytime/evening travel hours
practical_traffic = hourly_traffic[
    hourly_traffic["hour"].between(
        PRACTICAL_START_HOUR,
        PRACTICAL_END_HOUR
    )
].copy()

# Overall historical average for each day type
day_type_average = (
    df.groupby("day_type")["traffic_volume"]
      .mean()
      .rename("day_type_avg_volume")
)

# Add the corresponding day-type average
practical_traffic = practical_traffic.merge(
    day_type_average,
    on="day_type",
    how="left"
)

# Calculate percentage reduction relative to the
# overall historical average for that day type
practical_traffic["traffic_reduction_pct"] = (
    (
        practical_traffic["day_type_avg_volume"]
        - practical_traffic["avg_traffic_volume"]
    )
    / practical_traffic["day_type_avg_volume"]
    * 100
)

# Select the five lowest-traffic practical hours
best_practical_periods = (
    practical_traffic
    .sort_values(
        ["day_type", "avg_traffic_volume"]
    )
    .groupby("day_type")
    .head(5)
    .reset_index(drop=True)
)

print(
    f"Lowest-traffic practical periods "
    f"({PRACTICAL_START_HOUR:02d}:00-"
    f"{PRACTICAL_END_HOUR:02d}:00):"
)
print()

for day_type in ["Weekday", "Weekend"]:

    print(f"{day_type}:")

    result = best_practical_periods[
        best_practical_periods["day_type"] == day_type
    ][
        [
            "hour",
            "avg_traffic_volume",
            "day_type_avg_volume",
            "traffic_reduction_pct",
            "observations"
        ]
    ].copy()

    result["avg_traffic_volume"] = (
        result["avg_traffic_volume"].round(1)
    )

    result["day_type_avg_volume"] = (
        result["day_type_avg_volume"].round(1)
    )

    result["traffic_reduction_pct"] = (
        result["traffic_reduction_pct"].round(1)
    )

    print(result.to_string(index=False))
    print()

Lowest-traffic practical periods (06:00-22:00):

Weekday:
 hour  avg_traffic_volume  day_type_avg_volume  traffic_reduction_pct  observations
   22              2125.9               3533.3                   39.8          1427
   21              2673.0               3533.3                   24.3          1425
   20              2842.4               3533.3                   19.6          1418
   19              3297.1               3533.3                    6.7          1407
   10              4377.9               3533.3                  -23.9          1494

Weekend:
 hour  avg_traffic_volume  day_type_avg_volume  traffic_reduction_pct  observations
    6              1089.1               2570.7                   57.6           598
    7              1589.4               2570.7                   38.2           604
    8              2338.6               2570.7                    9.0           602
   22              2384.4               2570.7                    7.2           567
   21   

### Interpretation of Practical Travel Windows

Restricting the analysis to the practical 06:00–22:00 travel period produces different recommendations for weekdays and weekends.

For **weekdays**, 22:00 has the lowest historical average traffic volume within the practical window, at approximately **2,125.9 vehicles per hour**. This is approximately **39.8% lower than the overall weekday average of 3,533.3 vehicles per hour**. The next-best periods are 21:00 and 20:00, with traffic volumes approximately 24.3% and 19.6% below the weekday average, respectively.

For **weekends**, 06:00 has the lowest historical average traffic volume within the practical window, at approximately **1,089.1 vehicles per hour**. This is approximately **57.6% lower than the overall weekend average of 2,570.7 vehicles per hour**. 07:00 is also relatively favourable, with traffic approximately 38.2% below the weekend average.

The results demonstrate that suitable travel times differ substantially between weekdays and weekends. They also show why recommendations should be based on the observed data rather than on a fixed example time. For example, weekday traffic at 10:00 is approximately 23.9% above the overall weekday average in this dataset and would therefore not be an appropriate low-traffic recommendation.

These findings provide a quantitative basis for the recommendation system. The final system will use day type, the user's preferred travel window and, where appropriate, weather conditions to generate more personalised and actionable travel-time recommendations.

## 5.5 Weather-Aware Traffic Analysis

Weather conditions may influence traffic behaviour and should therefore be considered where sufficient historical data are available.

Before incorporating weather into the recommendation logic, the distribution of weather categories is examined. This prevents recommendations from relying on weather conditions with very limited historical support.

The analysis will:

1. Examine the available `weather_main` categories.
2. Measure the number of historical observations for each weather condition.
3. Compare traffic patterns across sufficiently represented weather categories.
4. Use weather as an optional recommendation condition rather than assuming that weather always has the same effect on traffic.

This approach keeps the recommendation system data-driven while reducing the risk of producing recommendations from sparsely represented weather conditions.

In [8]:
# ============================================================
# 5.5 Weather-Aware Traffic Analysis
# ============================================================

weather_summary = (
    df.groupby("weather_main")
      .agg(
          observations=("traffic_volume", "size"),
          avg_traffic_volume=("traffic_volume", "mean")
      )
      .sort_values("observations", ascending=False)
      .reset_index()
)

weather_summary["avg_traffic_volume"] = (
    weather_summary["avg_traffic_volume"].round(1)
)

weather_summary["data_share_pct"] = (
    weather_summary["observations"]
    / len(df)
    * 100
).round(1)

print("Historical weather distribution and traffic volume:")
print()
print(weather_summary.to_string(index=False))

Historical weather distribution and traffic volume:

weather_main  observations  avg_traffic_volume  data_share_pct
      Clouds         15158              3618.0            31.5
       Clear         13384              3055.6            27.8
        Mist          5949              2933.3            12.3
        Rain          5672              3317.9            11.8
        Snow          2875              3016.3             6.0
     Drizzle          1820              3292.2             3.8
        Haze          1360              3502.1             2.8
Thunderstorm          1033              2999.4             2.1
         Fog           912              2703.7             1.9
       Smoke            20              3237.6             0.0
      Squall             4              2061.8             0.0


## Interpretation of Weather Coverage

The dataset contains substantial historical coverage for several common weather conditions.

The most frequently observed categories are **Clouds (31.5%)**, **Clear (27.8%)**, **Mist (12.3%)**, and **Rain (11.8%)**. Snow also has meaningful historical coverage at approximately 6.0% of the dataset.

Average traffic volume differs across weather categories. However, these averages should not be interpreted as evidence that weather alone causes changes in traffic volume because weather conditions are also associated with different times, seasons and travel patterns.

Some rare categories have very limited historical support. In particular, **Smoke contains only 20 observations** and **Squall only 4 observations**. Recommendations based on such small samples could be unstable.

Therefore, the recommendation system will incorporate weather only when sufficient matching historical observations are available. If a requested weather condition has insufficient historical support, the system will fall back to the broader day-type recommendation and clearly inform the user.

This provides a more robust recommendation strategy than treating every weather category as equally reliable.

### 5.5.1 Weather-Specific Travel Patterns

To generate weather-aware recommendations, historical traffic is analysed jointly by:

- Day type (Weekday or Weekend)
- Weather condition
- Hour of day

Only the practical travel period from 06:00 to 22:00 is considered at this stage.

A minimum historical support threshold is also applied. A weather-specific hourly period must contain at least **30 observations** before it can be considered for a recommendation. This threshold is used as a practical safeguard against recommendations based on very small samples; it is not intended as a universal statistical rule.

The analysis identifies the lowest-traffic practical period for each sufficiently represented combination of day type and weather.

In [9]:
# ============================================================
# 5.5.1 Weather-Specific Travel Patterns
# ============================================================

MIN_WEATHER_OBSERVATIONS = 30

# Analyse practical hours only
weather_hourly = (
    df[
        df["hour"].between(
            PRACTICAL_START_HOUR,
            PRACTICAL_END_HOUR
        )
    ]
    .groupby(
        ["day_type", "weather_main", "hour"],
        as_index=False
    )
    .agg(
        avg_traffic_volume=("traffic_volume", "mean"),
        observations=("traffic_volume", "size")
    )
)

# Keep combinations with sufficient historical support
weather_hourly_supported = weather_hourly[
    weather_hourly["observations"] >= MIN_WEATHER_OBSERVATIONS
].copy()

# Find the lowest-traffic supported hour
# for each day type and weather condition
best_weather_periods = (
    weather_hourly_supported
    .sort_values(
        [
            "day_type",
            "weather_main",
            "avg_traffic_volume"
        ]
    )
    .groupby(
        ["day_type", "weather_main"],
        as_index=False
    )
    .first()
)

best_weather_periods["avg_traffic_volume"] = (
    best_weather_periods["avg_traffic_volume"].round(1)
)

print(
    "Best supported practical travel hour "
    "by day type and weather:"
)
print(
    f"Minimum observations per hourly condition: "
    f"{MIN_WEATHER_OBSERVATIONS}"
)
print()

print(
    best_weather_periods[
        [
            "day_type",
            "weather_main",
            "hour",
            "avg_traffic_volume",
            "observations"
        ]
    ].to_string(index=False)
)

Best supported practical travel hour by day type and weather:
Minimum observations per hourly condition: 30

day_type weather_main  hour  avg_traffic_volume  observations
 Weekday        Clear    22              2222.5           486
 Weekday       Clouds    22              2146.5           411
 Weekday      Drizzle    22              1997.8            65
 Weekday          Fog     9              4968.8            39
 Weekday         Haze    22              2157.4            31
 Weekday         Mist    22              1992.8           134
 Weekday         Rain    22              1998.7           174
 Weekday         Snow    22              2076.8            85
 Weekday Thunderstorm    21              2562.2            32
 Weekend        Clear     6              1116.6           171
 Weekend       Clouds     6              1100.5           133
 Weekend         Mist     6              1079.6           116
 Weekend         Rain     6              1054.8            67
 Weekend         Snow  

### Interpretation of Weather-Specific Travel Patterns

The weather-specific analysis shows that the preferred practical travel period remains relatively consistent for the most common weather conditions, but important differences and data-coverage limitations are also visible.

For **weekdays**, 22:00 is the lowest-traffic supported practical hour for several well-represented conditions, including Clear, Clouds, Mist, Rain and Snow. Historical average traffic at this hour is approximately 1,993–2,223 vehicles per hour across these conditions.

For **weekends**, 06:00 is consistently identified as the lowest-traffic supported practical hour for Clear, Clouds, Mist, Rain and Snow, with historical averages of approximately 959–1,117 vehicles per hour.

Some less common conditions provide weaker evidence. For example, the weekday Fog result is based on only 39 observations and has a relatively high average traffic volume of approximately 4,969 vehicles per hour. Thunderstorm also has limited hourly support.

These findings indicate that weather can be incorporated into the recommendation system, but the reliability of a weather-specific recommendation depends on the amount of matching historical data.

Therefore, the final recommendation function will:

- Use day type and the user's preferred travel window as the primary recommendation criteria.
- Apply a weather filter when the requested weather condition has sufficient historical support.
- Require a minimum number of matching observations before accepting a weather-specific recommendation.
- Fall back to a broader day-type recommendation when weather-specific evidence is insufficient.
- Report the historical traffic estimate and supporting sample size so that the recommendation remains transparent and evidence-based.

## 5.6 Traffic Timing Recommendation System

The analyses above are combined into a practical travel-timing recommendation system.

Because the dataset represents a **single traffic corridor**, the system recommends **when to travel rather than which route to take**.

The user can specify:

- Day type: Weekday or Weekend
- Preferred earliest travel hour
- Preferred latest travel hour
- Weather condition: optional

The recommendation algorithm:

1. Validates the user's inputs.
2. Filters historical observations by day type and preferred travel window.
3. Applies the requested weather condition when sufficient historical evidence is available.
4. Calculates average traffic volume for each eligible hour.
5. Requires sufficient historical observations for weather-specific recommendations.
6. Selects the lowest-traffic supported hour.
7. Verifies that the selected period is actually below the relevant baseline traffic level.
8. Falls back to a broader day-type recommendation when a weather-specific result is unsupported or does not represent lower traffic.
9. Generates a clear, plain-language recommendation for the user.

Internal processing, validation and fallback decisions are recorded using Python logging. Only the final user-facing recommendation is printed.

This design prevents the system from recommending a period merely because it is the lowest value within a small subgroup. A recommended period must also represent a genuine reduction relative to the relevant historical traffic baseline.

In [14]:
# ============================================================
# 5.6 Traffic Timing Recommendation System
# ============================================================

import logging

# ------------------------------------------------------------
# Logging configuration
# ------------------------------------------------------------

logger = logging.getLogger("traffic_recommendation")
logger.setLevel(logging.INFO)
logger.propagate = False

# Prevent duplicate handlers if this cell is re-run
for handler in logger.handlers[:]:
    handler.close()
    logger.removeHandler(handler)

formatter = logging.Formatter(
    "%(asctime)s - %(levelname)s - %(name)s - %(message)s"
)

# Console logging
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.INFO)
console_handler.setFormatter(formatter)

# File logging
file_handler = logging.FileHandler(
    "traffic_recommendation.log",
    mode="w"
)
file_handler.setLevel(logging.DEBUG)
file_handler.setFormatter(formatter)

logger.addHandler(console_handler)
logger.addHandler(file_handler)


# ------------------------------------------------------------
# Helper function
# ------------------------------------------------------------

def format_hour(hour):
    """Format an integer hour as HH:00."""
    return f"{int(hour):02d}:00"


# ------------------------------------------------------------
# Recommendation function
# ------------------------------------------------------------

def recommend_travel_window(
    df,
    day_type,
    start_hour=6,
    end_hour=22,
    weather=None,
    min_weather_observations=30
):
    """
    Recommend a lower-traffic one-hour travel window.

    The recommendation is based on historical traffic volume,
    day type, the user's preferred travel period and, where
    sufficiently supported, weather conditions.
    """

    logger.info(
        "Recommendation requested: "
        "day_type=%s, window=%s-%s, weather=%s",
        day_type,
        start_hour,
        end_hour,
        weather
    )

    # --------------------------------------------------------
    # 1. Validate inputs
    # --------------------------------------------------------

    valid_day_types = {
        "weekday": "Weekday",
        "weekend": "Weekend"
    }

    day_key = str(day_type).strip().lower()

    if day_key not in valid_day_types:
        logger.error(
            "Invalid day type: %s",
            day_type
        )
        raise ValueError(
            "day_type must be 'Weekday' or 'Weekend'."
        )

    selected_day_type = valid_day_types[day_key]

    if not isinstance(start_hour, (int, np.integer)):
        logger.error(
            "Invalid start_hour: %s",
            start_hour
        )
        raise TypeError(
            "start_hour must be an integer."
        )

    if not isinstance(end_hour, (int, np.integer)):
        logger.error(
            "Invalid end_hour: %s",
            end_hour
        )
        raise TypeError(
            "end_hour must be an integer."
        )

    if not 0 <= start_hour <= 22:
        logger.error(
            "start_hour outside valid range: %s",
            start_hour
        )
        raise ValueError(
            "start_hour must be between 0 and 22."
        )

    if not 0 <= end_hour <= 22:
        logger.error(
            "end_hour outside valid range: %s",
            end_hour
        )
        raise ValueError(
            "end_hour must be between 0 and 22."
        )

    if start_hour > end_hour:
        logger.error(
            "Invalid travel window: start=%s, end=%s",
            start_hour,
            end_hour
        )
        raise ValueError(
            "start_hour must be less than or equal to end_hour."
        )

    # --------------------------------------------------------
    # 2. Build day-type dataset and baseline
    # --------------------------------------------------------

    day_data = df[
        df["day_type"] == selected_day_type
    ].copy()

    if day_data.empty:
        logger.error(
            "No historical data available for %s.",
            selected_day_type
        )
        raise ValueError(
            "No historical observations are available "
            "for the selected day type."
        )

    # Baseline used to determine whether traffic is genuinely low
    day_type_average = (
        day_data["traffic_volume"].mean()
    )

    # Apply the user's preferred travel window
    base_data = day_data[
        day_data["hour"].between(
            start_hour,
            end_hour
        )
    ].copy()

    if base_data.empty:
        logger.error(
            "No data for %s in selected window.",
            selected_day_type
        )
        raise ValueError(
            "No historical observations match the selected "
            "day type and travel window."
        )

    recommendation_data = base_data

    weather_used = False
    fallback_used = False
    requested_weather = None

    # --------------------------------------------------------
    # 3. Apply optional weather condition
    # --------------------------------------------------------

    if weather is not None:

        weather_lookup = {
            str(value).strip().lower(): value
            for value in df[
                "weather_main"
            ].dropna().unique()
        }

        weather_key = (
            str(weather).strip().lower()
        )

        if weather_key not in weather_lookup:
            logger.error(
                "Unknown weather condition: %s",
                weather
            )
            raise ValueError(
                "Requested weather condition is not "
                "available in the historical dataset."
            )

        requested_weather = (
            weather_lookup[weather_key]
        )

        weather_data = base_data[
            base_data["weather_main"]
            == requested_weather
        ].copy()

        # Calculate historical support for each hour
        weather_hourly = (
            weather_data
            .groupby(
                "hour",
                as_index=False
            )
            .agg(
                avg_traffic_volume=(
                    "traffic_volume",
                    "mean"
                ),
                observations=(
                    "traffic_volume",
                    "size"
                )
            )
        )

        # Require sufficient observations
        supported_weather = weather_hourly[
            weather_hourly["observations"]
            >= min_weather_observations
        ].copy()

        # IMPORTANT:
        # A weather-specific period must also be below
        # the overall average for that day type.
        supported_weather = supported_weather[
            supported_weather[
                "avg_traffic_volume"
            ] < day_type_average
        ].copy()

        if not supported_weather.empty:

            supported_hours = (
                supported_weather["hour"]
            )

            recommendation_data = weather_data[
                weather_data["hour"].isin(
                    supported_hours
                )
            ].copy()

            weather_used = True

            logger.info(
                "Weather filter accepted: %s",
                requested_weather
            )

        else:

            fallback_used = True

            logger.warning(
                "Weather-specific recommendation for %s "
                "was unsupported or not below the %s "
                "baseline. Falling back to day-type data.",
                requested_weather,
                selected_day_type
            )

    # --------------------------------------------------------
    # 4. Aggregate candidate hours
    # --------------------------------------------------------

    hourly_candidates = (
        recommendation_data
        .groupby(
            "hour",
            as_index=False
        )
        .agg(
            avg_traffic_volume=(
                "traffic_volume",
                "mean"
            ),
            observations=(
                "traffic_volume",
                "size"
            )
        )
    )

    # Only periods below the day-type average
    # can be recommended.
    lower_traffic_candidates = (
        hourly_candidates[
            hourly_candidates[
                "avg_traffic_volume"
            ] < day_type_average
        ]
        .copy()
    )

    # --------------------------------------------------------
    # 5. Handle case where no lower-traffic period exists
    # --------------------------------------------------------

    if lower_traffic_candidates.empty:

        logger.warning(
            "No below-average traffic period found "
            "for %s between %02d:00 and %02d:00.",
            selected_day_type,
            start_hour,
            end_hour
        )

        recommendation = (
            f"No clearly lower-traffic period was found "
            f"for a {selected_day_type.lower()} journey "
            f"between {format_hour(start_hour)} and "
            f"{format_hour(end_hour + 1)}."
        )

        return {
            "success": False,
            "day_type": selected_day_type,
            "recommendation": recommendation
        }

    # Rank eligible periods from lowest traffic upward
    lower_traffic_candidates = (
        lower_traffic_candidates
        .sort_values(
            [
                "avg_traffic_volume",
                "observations"
            ],
            ascending=[True, False]
        )
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # 6. Select the best supported hour
    # --------------------------------------------------------

    best = lower_traffic_candidates.iloc[0]

    recommended_hour = int(
        best["hour"]
    )

    recommended_volume = float(
        best["avg_traffic_volume"]
    )

    observations = int(
        best["observations"]
    )

    reduction_pct = (
        (
            day_type_average
            - recommended_volume
        )
        / day_type_average
        * 100
    )

    start_text = format_hour(
        recommended_hour
    )

    end_text = format_hour(
        recommended_hour + 1
    )

    # --------------------------------------------------------
    # 7. Natural-language weather wording
    # --------------------------------------------------------

    weather_phrase = ""

    if weather_used:

        weather_wording = {
            "Rain": "rainy",
            "Snow": "snowy",
            "Clear": "clear",
            "Clouds": "cloudy",
            "Mist": "misty",
            "Fog": "foggy",
            "Drizzle": "drizzly",
            "Haze": "hazy",
            "Thunderstorm": "thunderstorm"
        }

        condition_text = weather_wording.get(
            requested_weather,
            requested_weather.lower()
        )

        weather_phrase = (
            f" under {condition_text} conditions"
        )

    # --------------------------------------------------------
    # 8. Generate plain-language recommendation
    # --------------------------------------------------------

    recommendation = (
        f"For a {selected_day_type.lower()} journey"
        f"{weather_phrase}, consider travelling between "
        f"{start_text} and {end_text}, when historical "
        f"traffic volume is about {reduction_pct:.1f}% "
        f"below the {selected_day_type.lower()} average."
    )

    if fallback_used:

        recommendation += (
            f" The requested {requested_weather} condition "
            f"did not provide a sufficiently supported "
            f"below-average period, so the recommendation "
            f"uses the broader "
            f"{selected_day_type.lower()} traffic pattern."
        )

    # --------------------------------------------------------
    # 9. Log recommendation details
    # --------------------------------------------------------

    logger.info(
        "Recommendation generated: %s %s-%s, "
        "volume=%.1f, reduction=%.1f%%, "
        "observations=%d",
        selected_day_type,
        start_text,
        end_text,
        recommended_volume,
        reduction_pct,
        observations
    )

    # --------------------------------------------------------
    # 10. Return transparent recommendation result
    # --------------------------------------------------------

    return {
        "success": True,
        "day_type": selected_day_type,
        "requested_weather": requested_weather,
        "weather_filter_used": weather_used,
        "fallback_used": fallback_used,
        "recommended_window": (
            f"{start_text} - {end_text}"
        ),
        "expected_traffic_volume": round(
            recommended_volume,
            1
        ),
        "day_type_average": round(
            day_type_average,
            1
        ),
        "traffic_reduction_pct": round(
            reduction_pct,
            1
        ),
        "supporting_observations": observations,
        "recommendation": recommendation
    }


# ------------------------------------------------------------
# System initialisation confirmation
# ------------------------------------------------------------

logger.info(
    "Traffic recommendation system "
    "initialised successfully."
)

2026-10-08 00:19:56,707 - INFO - traffic_recommendation - Traffic recommendation system initialised successfully.


In [15]:
# ============================================================
# 5.7 Demonstration of the Recommendation System
# ============================================================

# Scenario 1: Weekday
weekday_result = recommend_travel_window(
    df=df,
    day_type="Weekday",
    start_hour=6,
    end_hour=22
)

print(weekday_result["recommendation"])


# Scenario 2: Weekend
weekend_result = recommend_travel_window(
    df=df,
    day_type="Weekend",
    start_hour=6,
    end_hour=22
)

print(weekend_result["recommendation"])


# Scenario 3: Weekday with Rain
weekday_rain_result = recommend_travel_window(
    df=df,
    day_type="Weekday",
    start_hour=6,
    end_hour=22,
    weather="Rain"
)

print(weekday_rain_result["recommendation"])

2026-10-08 00:20:10,028 - INFO - traffic_recommendation - Recommendation requested: day_type=Weekday, window=6-22, weather=None
2026-10-08 00:20:10,131 - INFO - traffic_recommendation - Recommendation generated: Weekday 22:00-23:00, volume=2125.9, reduction=39.8%, observations=1427
2026-10-08 00:20:10,137 - INFO - traffic_recommendation - Recommendation requested: day_type=Weekend, window=6-22, weather=None
2026-10-08 00:20:10,172 - INFO - traffic_recommendation - Recommendation generated: Weekend 06:00-07:00, volume=1089.1, reduction=57.6%, observations=598
2026-10-08 00:20:10,176 - INFO - traffic_recommendation - Recommendation requested: day_type=Weekday, window=6-22, weather=Rain
2026-10-08 00:20:10,250 - INFO - traffic_recommendation - Weather filter accepted: Rain
2026-10-08 00:20:10,263 - INFO - traffic_recommendation - Recommendation generated: Weekday 22:00-23:00, volume=1998.7, reduction=43.4%, observations=174


For a weekday journey, consider travelling between 22:00 and 23:00, when historical traffic volume is about 39.8% below the weekday average.
For a weekend journey, consider travelling between 06:00 and 07:00, when historical traffic volume is about 57.6% below the weekend average.
For a weekday journey under rainy conditions, consider travelling between 22:00 and 23:00, when historical traffic volume is about 43.4% below the weekday average.


### Interpretation of Recommendation Results

The recommendation system successfully generates practical, plain-language travel advice for different travel scenarios.

For a **weekday journey between 06:00 and 22:00**, the system recommends travelling between **22:00 and 23:00**, when historical traffic volume is approximately **39.8% below the overall weekday average**.

For a **weekend journey**, the recommended period is **06:00 to 07:00**, when historical traffic volume is approximately **57.6% below the overall weekend average**.

When **rainy conditions** are specified for a weekday journey, the system recommends **22:00 to 23:00**. Historical traffic volume during this period is approximately **43.4% below the weekday average**, based on **174 historical observations**.

These results demonstrate that the system does not rely on a fixed travel-time rule. Instead, recommendations are generated from observed historical traffic patterns while considering day type, the user's preferred travel period and weather conditions when sufficient evidence is available.

The recommendations are therefore actionable, data-driven and expressed in plain language for non-technical users.

These recommendations describe historical traffic patterns on a single corridor. They are not route choices, real-time forecasts or causal claims.

## 5.8 Robustness and Fallback Testing

A practical recommendation system should not generate misleading advice when historical evidence is weak or when an input is invalid.

Two robustness checks are therefore performed:

1. **Weather fallback test:** Weekday Fog is tested because the previously identified supported fog period has relatively high traffic volume. The system should reject a weather-specific recommendation that is not below the weekday baseline and fall back to the broader weekday traffic pattern.

2. **Input validation test:** An invalid day type is supplied to confirm that the system detects incorrect user input and records the problem through logging rather than silently generating a recommendation.

These tests demonstrate that the recommendation system includes safeguards against unsupported or inappropriate recommendations.

In [17]:
# ============================================================
# 5.8 Robustness and Fallback Testing
# ============================================================

# ------------------------------------------------------------
# Test 1: Weather fallback
# ------------------------------------------------------------

fog_result = recommend_travel_window(
    df=df,
    day_type="Weekday",
    start_hour=6,
    end_hour=22,
    weather="Fog"
)

print(fog_result["recommendation"])


# ------------------------------------------------------------
# Test 2: Invalid input handling
# ------------------------------------------------------------

try:
    invalid_result = recommend_travel_window(
        df=df,
        day_type="Holiday",
        start_hour=6,
        end_hour=22
    )

    print(invalid_result["recommendation"])

except (ValueError, TypeError) as error:
    logger.info(
        "Input validation test passed: %s",
        error
    )

2026-10-08 00:34:47,873 - INFO - traffic_recommendation - Recommendation requested: day_type=Weekday, window=6-22, weather=Fog
2026-10-08 00:34:48,115 - WARNING - traffic_recommendation - Weather-specific recommendation for Fog was unsupported or not below the Weekday baseline. Falling back to day-type data.
2026-10-08 00:34:48,136 - INFO - traffic_recommendation - Recommendation generated: Weekday 22:00-23:00, volume=2125.9, reduction=39.8%, observations=1427
2026-10-08 00:34:48,143 - INFO - traffic_recommendation - Recommendation requested: day_type=Holiday, window=6-22, weather=None
2026-10-08 00:34:48,144 - ERROR - traffic_recommendation - Invalid day type: Holiday
2026-10-08 00:34:48,145 - INFO - traffic_recommendation - Input validation test passed: day_type must be 'Weekday' or 'Weekend'.


For a weekday journey, consider travelling between 22:00 and 23:00, when historical traffic volume is about 39.8% below the weekday average. The requested Fog condition did not provide a sufficiently supported below-average period, so the recommendation uses the broader weekday traffic pattern.


### Interpretation of Robustness Tests

The robustness tests confirm that the recommendation system does not automatically accept every weather-specific result.

For the **Weekday + Fog** scenario, the available fog-specific period did not represent traffic below the overall weekday baseline. The system therefore rejected the weather-specific result and correctly fell back to the broader weekday pattern, recommending **22:00 to 23:00**, when historical traffic volume is approximately **39.8% below the weekday average**.

The invalid `Holiday` day-type test was also correctly rejected because the system accepts only **Weekday** or **Weekend** as valid day types. The error was recorded through logging rather than producing an unsupported recommendation.

These tests demonstrate that the system includes safeguards for insufficient or inappropriate evidence, input validation and transparent fallback behaviour. This reduces the risk of presenting a statistically supported but practically misleading recommendation to the user.

## 5.9 Limitations

The recommendation system provides practical travel-timing guidance, but several limitations should be considered when interpreting its outputs.

### Single-corridor coverage
The dataset represents traffic observations from a single westbound I-94 corridor. Therefore, the system recommends **travel timing rather than alternative routes** and should not be interpreted as a city-wide route optimisation system.

### Historical rather than real-time recommendations
Recommendations are derived from historical traffic patterns. The system does not currently use live traffic feeds, incidents, roadworks, special events or real-time weather forecasts. Unexpected current conditions may therefore differ from the historical recommendation.

### Weather data limitations
Weather-specific recommendations depend on the amount of historical evidence available. Rare conditions may contain too few observations to support a reliable recommendation. The system therefore applies a minimum observation threshold and falls back to the broader day-type pattern when appropriate.

### Association rather than causation
Differences in traffic volume across weather conditions should not be interpreted as evidence that weather directly causes those differences. Time of day, season, commuting behaviour, holidays and other factors may also influence traffic patterns.

### Recommendation baseline
A travel period is recommended only when its historical average traffic volume is below the overall average for the relevant day type. This provides a transparent and conservative decision rule, but it does not guarantee that the recommended period will have lower traffic on every individual journey.

### Practical travel window
The 06:00–22:00 period is used as a practical analytical default to avoid automatically recommending overnight minimum-traffic periods. The recommendation function nevertheless allows the user to specify a different preferred travel window.

Overall, the system should be interpreted as a **historical decision-support tool**, not as a real-time navigation or traffic forecasting service.

## 5.10 Final Validation

The final validation checks whether the Task 5 recommendation system satisfies the required functional and analytical criteria.

The validation confirms that the system:

- Uses the engineered traffic dataset from the earlier project pipeline.
- Identifies historically lower-traffic travel periods.
- Distinguishes between weekday and weekend traffic patterns.
- Uses a practical user-defined travel window.
- Incorporates weather when sufficient historical evidence is available.
- Rejects weather-specific periods that are not below the relevant day-type baseline.
- Falls back safely when weather-specific evidence is insufficient or inappropriate.
- Generates actionable plain-language recommendations.
- Provides supporting traffic reduction and observation information.
- Uses logging for internal system activity and validation.

In [19]:
# ============================================================
# 5.10 Final Validation
# ============================================================

validation_results = {}


# ------------------------------------------------------------
# 1. Required dataset features
# ------------------------------------------------------------

required_task5_columns = {
    "traffic_volume",
    "hour",
    "day_type",
    "is_weekend",
    "weather_main"
}

validation_results["Required features available"] = (
    required_task5_columns.issubset(df.columns)
)


# ------------------------------------------------------------
# 2. Lower-traffic periods identified
# ------------------------------------------------------------

validation_results["Lower-traffic periods identified"] = (
    not lowest_traffic_periods.empty
)


# ------------------------------------------------------------
# 3. Weekday and weekend supported
# ------------------------------------------------------------

validation_results["Day types supported"] = (
    {"Weekday", "Weekend"}
    .issubset(set(df["day_type"].dropna().unique()))
)


# ------------------------------------------------------------
# 4. Practical travel-window analysis completed
# ------------------------------------------------------------

validation_results["Practical travel window analysed"] = (
    not best_practical_periods.empty
)


# ------------------------------------------------------------
# 5. Weather-aware analysis completed
# ------------------------------------------------------------

validation_results["Weather-aware analysis completed"] = (
    not best_weather_periods.empty
)


# ------------------------------------------------------------
# 6. Standard recommendations generated successfully
# ------------------------------------------------------------

validation_results["Weekday recommendation generated"] = (
    weekday_result.get("success", False)
)

validation_results["Weekend recommendation generated"] = (
    weekend_result.get("success", False)
)

validation_results["Rain recommendation generated"] = (
    weekday_rain_result.get("success", False)
)


# ------------------------------------------------------------
# 7. Recommended periods are below day-type baselines
# ------------------------------------------------------------

standard_results = [
    weekday_result,
    weekend_result,
    weekday_rain_result
]

validation_results[
    "Recommendations below day-type baseline"
] = all(
    result.get("traffic_reduction_pct", 0) > 0
    for result in standard_results
)


# ------------------------------------------------------------
# 8. Weather-specific recommendation is supported
# ------------------------------------------------------------

validation_results[
    "Weather recommendation sufficiently supported"
] = (
    weekday_rain_result.get(
        "weather_filter_used",
        False
    )
    and weekday_rain_result.get(
        "supporting_observations",
        0
    ) >= MIN_WEATHER_OBSERVATIONS
)


# ------------------------------------------------------------
# 9. Fog fallback works correctly
# ------------------------------------------------------------

validation_results["Weather fallback works"] = (
    fog_result.get("success", False)
    and fog_result.get("fallback_used", False)
    and not fog_result.get(
        "weather_filter_used",
        False
    )
)


# ------------------------------------------------------------
# 10. Plain-language recommendations generated
# ------------------------------------------------------------

validation_results[
    "Plain-language recommendations generated"
] = all(
    isinstance(
        result.get("recommendation"),
        str
    )
    and len(
        result.get("recommendation", "")
    ) > 0
    for result in standard_results
)


# ------------------------------------------------------------
# Log validation results
# ------------------------------------------------------------

logger.info("Task 5 final validation started.")

for check, passed in validation_results.items():

    if passed:
        logger.info(
            "PASS - %s",
            check
        )
    else:
        logger.error(
            "FAIL - %s",
            check
        )


# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

task5_passed = all(
    validation_results.values()
)

if task5_passed:

    logger.info(
        "TASK 5 STATUS: PASS - "
        "All required checks completed."
    )

else:

    logger.error(
        "TASK 5 STATUS: FAIL - "
        "One or more required checks failed."
    )

2026-10-08 00:38:53,632 - INFO - traffic_recommendation - Task 5 final validation started.
2026-10-08 00:38:53,636 - INFO - traffic_recommendation - PASS - Required features available
2026-10-08 00:38:53,637 - INFO - traffic_recommendation - PASS - Lower-traffic periods identified
2026-10-08 00:38:53,639 - INFO - traffic_recommendation - PASS - Day types supported
2026-10-08 00:38:53,640 - INFO - traffic_recommendation - PASS - Practical travel window analysed
2026-10-08 00:38:53,642 - INFO - traffic_recommendation - PASS - Weather-aware analysis completed
2026-10-08 00:38:53,643 - INFO - traffic_recommendation - PASS - Weekday recommendation generated
2026-10-08 00:38:53,644 - INFO - traffic_recommendation - PASS - Weekend recommendation generated
2026-10-08 00:38:53,646 - INFO - traffic_recommendation - PASS - Rain recommendation generated
2026-10-08 00:38:53,647 - INFO - traffic_recommendation - PASS - Recommendations below day-type baseline
2026-10-08 00:38:53,648 - INFO - traffic_